# Phase 5: Legal Statute Identification with the pre-trained **Transformer-XL + GQA** (decoder-only, hierarchical, multi-label)

This is the Transformer-XL twin of the encoder–decoder Phase-5 notebook. The data loading, label space, chunk-level head, losses,
thresholding, metrics, KD and submission format are the same, so the results are directly comparable. Only the backbone differs, and its
code (model class, memory-lean attention, PEFT modules) comes from your Transformer-XL Phase-7 notebook.

```
document → chunks  <s> + 510 BPE tokens + </s>   (the Phase-4/7 Transformer-XL window format)
        → Phase-1 Transformer-XL (adapted)     → token states (causal: token t sees tokens ≤ t)
        → chunk vector = [ mean | max | last ]   (any subset; 'last' = the </s> state, the only one that has read the whole chunk)
        → chunk-level BiGRU / BiLSTM / small BERT-style encoder  (cross-chunk context; this is where the bidirectionality comes from)
        → attention pooling → one sigmoid per statute
```

**What is specific to Transformer-XL (and why)**

| | Encoder–decoder notebook | This notebook |
|---|---|---|
| backbone direction | bidirectional encoder | **causal**. The `</s>` (`last`) state is the only one that has seen the whole chunk, so `last` is on by default |
| chunk summary | `mean \| max \| eos \| dec` | `mean \| max \| last` |
| chunk format | 511 + `<eos>` | `<s>` + 510 + `</s>` (as in your TXL Phase-4/7) |
| attention | fused SDPA + T5 bias | **lean Transformer-XL attention**: `(q+u)·k` inside SDPA, with `shift((q+v)·r)` passed as the additive mask (self-tested against the Phase-1 code) |
| prefix tuning | learned K/V prefixes | **learned Transformer-XL memory** in each block, fed through the model's own `memory` slot and initialised from real layer inputs |
| segment recurrence | n/a | **not used**: chunks are encoded independently, so chunk order is handled by the BiGRU. Say so in the report |
| KD student | first 4 encoder layers | **first 4 of 8 blocks** (a coherent truncated network; output re-normalised with a parameter-free LayerNorm) |

**The six experiments (brief §11.2)**: all start from the Phase-1 checkpoint, and no external model is used.

| # | experiment | technique | trained |
|---|---|---|---|
| E1 | `e1_lorma_plus` | **LoRMA⁺**, multiplicative `(I + α/r·BA)·W₀` (compulsory) | `A`, `B` on `q_proj`, `v_proj` of the top 4 blocks |
| E2 | `e2_adalora` | **AdaLoRA** (SVD form + importance-based rank pruning) | `P`, `Λ`, `Q` on all 6 linears of the top 4 blocks, rank 12 → 6 |
| E3 | `e3_qlora` | **QLoRA** (all 8 blocks stored in 4-bit NF4, LoRA on top) | LoRA `A`, `B` on the top 4 blocks |
| E4 | `e4_adapter` | **Houlsby adapters** (after attention and after the FFN) | adapters of the top 4 blocks |
| E5 | `e5_partial_ft` | **Partial fine-tuning** | top 3 blocks + final norm |
| E6 | `e6_kd_student` | **Knowledge distillation** | 4-block student + small BiGRU; the teacher is the ensemble of E1–E5 |

Optional extras in the RUN cell: prefix tuning (XL memory), prompt tuning, full fine-tuning, LoRA, LoRMA-π, and the student without a teacher.
In every PEFT method, only the top `top_layers` blocks get gradients. The blocks below run under `no_grad`, which keeps token-mode training feasible on a T4.

> All hyper-parameters in the RUN cell are **starting points** that make the notebook run end to end. Tuning them and interpreting the
> results is your job (course rules §17.5; declare any LLM use). Run one experiment first and read its `s/batch → ~N min per epoch` line.

**Run order:** top to bottom. Needs the Phase-1 Transformer-XL checkpoint and tokenizer (same paths as your TXL Phase-7 notebook) and an `HF_TOKEN` Colab secret.

In [1]:
# ============================================================
# 0. SETUP
# ============================================================
import os, sys, re, json, math, time, random, copy, pickle, glob, shutil, gc, unicodedata, zipfile, hashlib
from types import SimpleNamespace
from contextlib import nullcontext
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint as grad_ckpt

from google.colab import drive
drive.mount("/content/drive")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "no GPU")
AMP_DTYPE = (None if device.type != "cuda" else
             torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16)   # re-checked by the self-test

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
print("HF token set:", bool(HF_TOKEN))

# ---- Phase-1 Transformer-XL + GQA artefacts (same paths as your TXL Phase-7 notebook; edit if yours differ) ----
P1_DIR = "/content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache"
CHECKPOINT_PATH = os.path.join(P1_DIR, "transformer_xl_gqa_checkpoints", "best_model_transformer_xl_gqa-small.pt")
TOKENIZER_PATH = os.path.join(P1_DIR, "final_bpe_en.json")
MODEL_CONFIG = {"vocab_size": 32768, "d_model": 768, "num_layers": 8, "num_query_heads": 32, "num_kv_heads": 8,
                "memory_length": 128, "dropout": 0.1}

p5_dir = "/content/drive/MyDrive/Phase5_LSI_TXL"; os.makedirs(p5_dir, exist_ok=True)
FEAT_LOCAL = "/content/feat_cache_txl"; os.makedirs(FEAT_LOCAL, exist_ok=True)      # local disk: much faster than Drive

Mounted at /content/drive
device: cuda | Tesla T4
HF token set: True


## 1. Phase-1 tokenizer and text normalisation (identical to the TXL Phase-4/7 notebooks)

In [2]:
import os, re, json, heapq, collections

try:
    import psutil
    _PROC = psutil.Process(os.getpid())
except Exception:
    _PROC = None

def _ram_gb():
    return _PROC.memory_info().rss / (1024 ** 3) if _PROC else 0.0

WORD_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)
_WORKER = None
def _init_worker(tok):
    global _WORKER
    _WORKER = tok
def _encode_chunk(args):
    texts, add_special = args
    return [_WORKER.encode(t, add_special) for t in texts]

class BPETokenizer:
    _HEAP_THRESHOLD = 24
    _CACHE_LIMIT = 500_000

    def __init__(self, vocab_size, end_marker="</w>", reserved_tokens=None):
        self.vocab_size = vocab_size
        self.end_marker = end_marker
        self.reserved_tokens = reserved_tokens if reserved_tokens is not None else ["<pad>", "<unk>", "<s>", "</s>"]
        self.itos = list(self.reserved_tokens)
        self.stoi = {t: i for i, t in enumerate(self.reserved_tokens)}
        self.merges = {}
        self._ready = False
        self._cache = {}

    @staticmethod
    def from_json(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = BPETokenizer(vocab_size=data["vocab_size"], end_marker=data.get("end_marker", "</w>"),
                            reserved_tokens=data.get("reserved_tokens", ["<pad>", "<unk>", "<s>", "</s>"]))
        tok.itos = data["itos"]; tok.stoi = data["stoi"]
        tok.merges = {tuple(p): r for p, r in data["merges"]}
        tok._ready = False; tok._cache = {}
        return tok

    def _prepare_inference(self):
        if self._ready: return
        self._byte_ids = [self.stoi[bytes([b]).decode("latin-1")] for b in range(256)]
        self._first_merge_id = len(self.itos) - len(self.merges)
        self._end_id = self.stoi.get(self.end_marker, self.stoi["<unk>"])
        self._bos = self.stoi["<s>"]; self._eos = self.stoi["</s>"]
        base = self._first_merge_id
        for (a, b), r in self.merges.items():
            if self.itos[base + r] != self.itos[a] + self.itos[b]:
                raise ValueError("Merge/vocab mismatch: cannot infer merge ids from ranks.")
        self._M = len(self.itos); M = self._M
        self._ranks = {a * M + b: r for (a, b), r in self.merges.items()}
        self._cache = {}; self._ready = True

    def _merge_small(self, ids):
        get, M, base = self._ranks.get, self._M, self._first_merge_id
        while len(ids) > 1:
            best, best_i = None, -1
            for i in range(len(ids) - 1):
                r = get(ids[i] * M + ids[i + 1])
                if r is not None and (best is None or r < best):
                    best, best_i = r, i
            if best is None: break
            a, b, new = ids[best_i], ids[best_i + 1], base + best
            out, i, n = [], 0, len(ids)
            while i < n:
                if i < n - 1 and ids[i] == a and ids[i + 1] == b:
                    out.append(new); i += 2
                else:
                    out.append(ids[i]); i += 1
            ids = out
        return ids

    def _merge_heap(self, ids):
        get, M, base = self._ranks.get, self._M, self._first_merge_id
        n = len(ids)
        nxt = list(range(1, n + 1)); nxt[-1] = -1
        prv = list(range(-1, n - 1)); alive = [True] * n
        heap = []
        for i in range(n - 1):
            r = get(ids[i] * M + ids[i + 1])
            if r is not None: heap.append((r, i))
        heapq.heapify(heap)
        while heap:
            r, i = heapq.heappop(heap)
            if not alive[i]: continue
            j = nxt[i]
            if j == -1 or get(ids[i] * M + ids[j]) != r: continue
            ids[i] = base + r; alive[j] = False
            k = nxt[j]; nxt[i] = k
            if k != -1: prv[k] = i
            p = prv[i]
            if p != -1:
                r2 = get(ids[p] * M + ids[i])
                if r2 is not None: heapq.heappush(heap, (r2, p))
            if k != -1:
                r2 = get(ids[i] * M + ids[k])
                if r2 is not None: heapq.heappush(heap, (r2, i))
        return [ids[i] for i in range(n) if alive[i]]

    def _bpe_word(self, word):
        hit = self._cache.get(word)
        if hit is not None: return hit
        ids = [self._byte_ids[b] for b in word.encode("utf-8")]
        ids.append(self._end_id)
        ids = self._merge_small(ids) if len(ids) <= self._HEAP_THRESHOLD else self._merge_heap(ids)
        result = tuple(ids)
        if len(self._cache) < self._CACHE_LIMIT: self._cache[word] = result
        return result

    def encode(self, text, add_special_tokens=True):
        self._prepare_inference()
        out = [self._bos] if add_special_tokens else []
        extend = out.extend; cache_get = self._cache.get; bpe = self._bpe_word
        for w in WORD_RE.findall(text):
            t = cache_get(w)
            if t is None: t = bpe(w)
            extend(t)
        if add_special_tokens: out.append(self._eos)
        return out

    def encode_batch(self, texts, add_special_tokens=True, num_workers=None, chunk_size=256):
        """Encode many documents, optionally across processes (fork only)."""
        self._prepare_inference()
        texts = list(texts)
        if num_workers is None: num_workers = os.cpu_count() or 1
        if num_workers <= 1 or len(texts) < 2 * chunk_size:
            return [self.encode(t, add_special_tokens) for t in texts]
        import multiprocessing as mp
        chunks = [(texts[i:i + chunk_size], add_special_tokens) for i in range(0, len(texts), chunk_size)]
        ctx = mp.get_context("fork")
        with ctx.Pool(num_workers, initializer=_init_worker, initargs=(self,)) as pool:
            results = list(pool.imap(_encode_chunk, chunks))
        return [ids for chunk in results for ids in chunk]

    def decode(self, token_ids):
        self._prepare_inference()
        filtered_ids = [tid for tid in token_ids if self.itos[tid] not in self.reserved_tokens]
        decoded_text = "".join([self.itos[tid] for tid in filtered_ids]).replace(self.end_marker, " ")
        return decoded_text.strip()

if not os.path.isfile(TOKENIZER_PATH): raise FileNotFoundError(TOKENIZER_PATH)
tokenizer = BPETokenizer.from_json(TOKENIZER_PATH)
PAD_ID = tokenizer.stoi.get("<pad>", 0); BOS_ID = tokenizer.stoi.get("<s>", 2); EOS_ID = tokenizer.stoi.get("</s>", 3)
VOCAB_SIZE = len(tokenizer.itos)
assert VOCAB_SIZE == MODEL_CONFIG["vocab_size"], "tokenizer vocab must match the Phase-1 checkpoint"
print("vocab", VOCAB_SIZE, "| PAD/BOS/EOS", PAD_ID, BOS_ID, EOS_ID)


def normalize_text(x):
    """the cleaning used by the Phase-4 Transformer-XL notebook (NFKC + whitespace collapse)"""
    s = unicodedata.normalize("NFKC", "" if x is None else str(x))
    return re.sub(r"\s+", " ", s.replace("\u00a0", " ")).strip()

vocab 32768 | PAD/BOS/EOS 0 2 3


## 2. Phase-1 Transformer-XL + GQA (architecture copied unchanged; the checkpoint must load with no missing/unexpected keys)

In [3]:
import math

class TransformerXLRelativePositionalEncoding(nn.Module):
    def __init__(self, d_model: int, base: float = 10000.0):
        super().__init__()
        if d_model % 2:
            raise ValueError("d_model must be even")
        self.d_model = d_model
        inv_freq = 1.0 / (base ** (torch.arange(0, d_model, 2, dtype=torch.float32) / d_model))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, query_length: int, key_length: int, device=None):
        if key_length <= 0:
            raise ValueError("key_length must be positive")
        if device is None:
            device = self.inv_freq.device
        pos_seq = torch.arange(key_length - 1, -1, -1, dtype=torch.float32, device=device)
        sinusoid_inp = torch.outer(pos_seq, self.inv_freq.to(device))
        return torch.cat([sinusoid_inp.sin(), sinusoid_inp.cos()], dim=-1)

class TransformerXLGQAAttention(nn.Module):
    def __init__(self, d_model, num_query_heads, num_kv_heads, dropout=0.0):
        super().__init__()
        if d_model % num_query_heads != 0:
            raise ValueError("d_model must be divisible by num_query_heads")
        if num_query_heads % num_kv_heads != 0:
            raise ValueError("num_query_heads must be divisible by num_kv_heads")
        self.d_model = d_model
        self.num_query_heads = num_query_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = d_model // num_query_heads
        self.group_size = num_query_heads // num_kv_heads

        self.q_proj = nn.Linear(d_model, num_query_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.r_proj = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.u = nn.Parameter(torch.zeros(num_query_heads, self.head_dim))
        self.v = nn.Parameter(torch.zeros(num_query_heads, self.head_dim))
        nn.init.normal_(self.u, std=0.02)
        nn.init.normal_(self.v, std=0.02)
        self.dropout = nn.Dropout(dropout)

    def _q_heads(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.num_query_heads, self.head_dim).transpose(1, 2)

    def _kv_heads(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)

    def _expand_kv(self, x):
        B, H, T, D = x.shape
        return x.unsqueeze(2).expand(B, H, self.group_size, T, D).reshape(B, self.num_query_heads, T, D)

    @staticmethod
    def _rel_shift(x):
        B, H, Q, K = x.shape
        zero = torch.zeros((B, H, Q, 1), dtype=x.dtype, device=x.device)
        x = torch.cat([zero, x], dim=-1)
        x = x.view(B, H, K + 1, Q)
        x = x[:, :, 1:].view(B, H, Q, K)
        return x

    @staticmethod
    def _causal_mask(query_length, key_length, device):
        memory_length = key_length - query_length
        return torch.triu(torch.ones((query_length, key_length), dtype=torch.bool, device=device),
                           diagonal=memory_length + 1)

    def forward(self, x, memory=None, relative_position=None):
        B, Q, _ = x.shape
        if memory is not None and memory.size(1) > 0:
            kv_input = torch.cat([memory, x], dim=1)
        else:
            kv_input = x
        K = kv_input.size(1)

        q = self._q_heads(self.q_proj(x))
        k = self._expand_kv(self._kv_heads(self.k_proj(kv_input)))
        v = self._expand_kv(self._kv_heads(self.v_proj(kv_input)))

        ac = torch.einsum("bhqd, bhkd->bhqk", q, k)
        ac = ac + torch.einsum("hd,bhkd->bhk", self.u, k).unsqueeze(2)

        if relative_position is None:
            raise ValueError("relative_position is required for Transformer-XL attention")

        r = self._expand_kv(self._kv_heads(self.r_proj(relative_position).unsqueeze(0)))
        bd = torch.einsum("bhqd,bhkd->bhqk", q, r)
        bd = bd + torch.einsum("hd,bhkd->bhk", self.v, r).unsqueeze(2)
        bd = self._rel_shift(bd)

        scores = (ac + bd) / math.sqrt(self.head_dim)
        mask = self._causal_mask(Q, K, x.device)
        scores = scores.masked_fill(mask.view(1, 1, Q, K), torch.finfo(scores.dtype).min)
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, Q, self.d_model)
        return self.out_proj(out)


class TransformerXLBlockGQA(nn.Module):
    def __init__(self, d_model, num_query_heads, num_kv_heads, dropout=0.0, ffn_mult=4):
        super().__init__()
        self.attn = TransformerXLGQAAttention(d_model, num_query_heads, num_kv_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        ffn_dim = ffn_mult * d_model
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model), nn.Dropout(dropout),
        )

    def forward(self, x, memory, relative_position):
        x = self.norm1(x + self.attn(x, memory=memory, relative_position=relative_position))
        x = self.norm2(x + self.ffn(x))
        return x


class TransformerXLGQAModel(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers, num_query_heads, num_kv_heads, memory_length,
                 dropout=0.0, tie_embeddings=True):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.num_layers = num_layers
        self.memory_length = memory_length
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = TransformerXLRelativePositionalEncoding(d_model)
        self.blocks = nn.ModuleList([
            TransformerXLBlockGQA(d_model, num_query_heads, num_kv_heads, dropout) for _ in range(num_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        if tie_embeddings:
            self.lm_head.weight = self.embedding.weight

    def forward(self, input_ids, memory=None, labels=None):
        x = self.embedding(input_ids)
        Q = x.size(1)
        if memory is None:
            memory = [None] * self.num_layers
        if len(memory) != self.num_layers:
            raise ValueError(f"memory must contain {self.num_layers} entries")
        mem_len = 0 if memory[0] is None else memory[0].size(1)
        K = mem_len + Q
        rel_pos = self.pos_encoding(Q, K, device=x.device)
        for i, block in enumerate(self.blocks):
            x = block(x, memory[i], rel_pos)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits.reshape(-1, self.vocab_size), labels.reshape(-1))
        return {"logits": logits, "loss": loss}


if not os.path.isfile(CHECKPOINT_PATH): raise FileNotFoundError(CHECKPOINT_PATH)
ck = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
state = ck.get("model_state_dict", ck.get("state_dict", ck)) if isinstance(ck, dict) else ck
if isinstance(ck, dict): print("checkpoint keys:", list(ck.keys())[:12], "| epoch", ck.get("epoch"), "| best_val_loss", ck.get("best_val_loss"))
txl = TransformerXLGQAModel(**MODEL_CONFIG)
missing, unexpected = txl.load_state_dict(state, strict=False)
if missing or unexpected:                      # hard error on any mismatch (same rule as the Phase-3 notebook)
    print("Missing:", missing[:10], "count =", len(missing)); print("Unexpected:", unexpected[:10], "count =", len(unexpected))
    raise RuntimeError("Checkpoint is not compatible with MODEL_CONFIG / the Transformer-XL architecture above.")
del ck, state; gc.collect()
txl.eval()
print(f"Loaded Phase-1 Transformer-XL + GQA: {sum(p.numel() for n, p in txl.named_parameters() if not n.startswith('lm_head'))/1e6:.1f}M params (LM head is tied to the embedding and dropped below)")

checkpoint keys: ['epoch', 'global_step', 'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'scaler_state_dict', 'best_val_loss', 'loss_history'] | epoch 7 | best_val_loss 3.077281383833674
Loaded Phase-1 Transformer-XL + GQA: 76.0M params (LM head is tied to the embedding and dropped below)


## 3. Memory-lean attention + the backbone used by this phase
The attention patch is the same as in Phase 7. `txl_hidden` gains one option for this phase: a **truncated** backbone, made of the first k
blocks. Its output goes through a parameter-free LayerNorm, because the Phase-1 `final_norm` was trained on the output of block 8.
That is the KD student.

In [4]:
if not hasattr(TransformerXLGQAAttention, "_orig_forward"):
    TransformerXLGQAAttention._orig_forward = TransformerXLGQAAttention.forward


def txl_attn_forward(self, x, memory=None, relative_position=None):
    """same maths as Phase 1: softmax([ (q+u).k + shift((q+v).r) ] / sqrt(d_h) + causal) v, via fused SDPA"""
    B, Q, _ = x.shape
    kv_input = torch.cat([memory.to(x.dtype), x], dim=1) if memory is not None and memory.size(1) > 0 else x
    K = kv_input.size(1)
    q = self._q_heads(self.q_proj(x))
    k = self._expand_kv(self._kv_heads(self.k_proj(kv_input)))
    v = self._expand_kv(self._kv_heads(self.v_proj(kv_input)))
    r = self._expand_kv(self._kv_heads(self.r_proj(relative_position.to(x.dtype)).unsqueeze(0)))      # [1,H,K,dh]
    qv = q + self.v.to(q.dtype)[None, :, None, :]
    bd = self._rel_shift(torch.einsum("bhqd,hkd->bhqk", qv, r[0]))
    scale = 1.0 / math.sqrt(self.head_dim)
    mask = self._causal_mask(Q, K, x.device)
    bias = (bd * scale).masked_fill(mask.view(1, 1, Q, K), float("-inf")).to(q.dtype)
    qu = q + self.u.to(q.dtype)[None, :, None, :]
    drop = self.dropout.p if self.training else 0.0
    out = F.scaled_dot_product_attention(qu, k, v, attn_mask=bias, dropout_p=drop, scale=scale)
    return self.out_proj(out.transpose(1, 2).reshape(B, Q, self.d_model))


TransformerXLGQAAttention.forward = txl_attn_forward


class TXLEncoder(nn.Module):
    """Phase-1 Transformer-XL embedding + blocks + final norm (LM head dropped).
    prefix  : optional learned per-block memory [n_blocks, P, d] (prefix tuning through the XL memory slot)
    out_norm: 'final' -> Phase-1 final_norm ; 'plain' -> parameter-free LayerNorm (truncated student)"""

    def __init__(self, g):
        super().__init__()
        self.embed = copy.deepcopy(g.embedding)
        self.blocks = nn.ModuleList([copy.deepcopy(b) for b in g.blocks])
        self.final_norm = copy.deepcopy(g.final_norm)
        self.d_model = self.embed.embedding_dim
        self.pos = TransformerXLRelativePositionalEncoding(self.d_model)
        self.prefix = None
        self.out_norm = "final"

    def truncate(self, n_blocks):
        if n_blocks is not None and n_blocks < len(self.blocks):
            self.blocks = nn.ModuleList(list(self.blocks)[:n_blocks])
            self.out_norm = "plain"
        return self


def txl_hidden(bb, ids=None, embeds=None, ckpt=False, return_layer_inputs=False):
    """right-padded chunks [N,L] (causal: padding never reaches real tokens) -> final states [N,L,d].
    Blocks below the first one with trainable parameters run under no_grad (no activation memory)."""
    x = bb.embed(ids) if embeds is None else embeds
    N, Q = x.shape[:2]
    PL = 0 if bb.prefix is None else bb.prefix.size(1)
    rel = bb.pos(Q, Q + PL, device=x.device)
    lo = 0
    if torch.is_grad_enabled() and not x.requires_grad and PL == 0:
        while lo < len(bb.blocks) and not any(p.requires_grad for p in bb.blocks[lo].parameters()):
            lo += 1
    ins = []
    for i, blk in enumerate(bb.blocks):
        if return_layer_inputs:
            ins.append(x)
        m = bb.prefix[i].unsqueeze(0).expand(N, -1, -1) if PL else None
        if i < lo:
            with torch.no_grad():
                x = blk(x, m, rel)
        elif ckpt and torch.is_grad_enabled():
            x = grad_ckpt(lambda h, b=blk, m=m: b(h, m, rel), x, use_reentrant=False)
        else:
            x = blk(x, m, rel)
    h = bb.final_norm(x) if getattr(bb, "out_norm", "final") == "final" else F.layer_norm(x, x.shape[-1:])
    return (h, ins) if return_layer_inputs else h


TXL_BASE = TXLEncoder(txl).eval()
N_BLOCKS = len(TXL_BASE.blocks)
D_MODEL = TXL_BASE.d_model
CKPT_SIG = hashlib.md5(f"{os.path.basename(CHECKPOINT_PATH)}_{os.path.getsize(CHECKPOINT_PATH)}".encode()).hexdigest()[:8]
del txl; gc.collect()
print(f"Transformer-XL backbone: {sum(p.numel() for p in TXL_BASE.parameters())/1e6:.1f}M params, {N_BLOCKS} blocks, d={D_MODEL}")


def txl_view(n_blocks=None):
    """light, NON-copying view of the frozen Phase-1 backbone (optionally truncated) for feature extraction"""
    n = N_BLOCKS if n_blocks is None else n_blocks
    return SimpleNamespace(embed=TXL_BASE.embed, blocks=TXL_BASE.blocks[:n], final_norm=TXL_BASE.final_norm,
                           pos=TXL_BASE.pos, prefix=None, out_norm="final" if n == N_BLOCKS else "plain")

Transformer-XL backbone: 76.0M params, 8 blocks, d=768


## 4. Phase-5 (LSI) data (same reading logic as the encoder–decoder notebook; statute ids keep their original type)

In [5]:
from huggingface_hub import hf_hub_download, HfApi

REPO_ID = "Exploration-Lab/CS-781-Capstone"
_REPO_FILES = HfApi().list_repo_files(REPO_ID, repo_type="dataset", token=HF_TOKEN)
_LABEL_KEYS = ("labels", "statutes", "statute_ids", "label")


def find_repo_file(basename):
    hits = [f for f in _REPO_FILES if f.endswith("/" + basename) and ("lsi" in f.lower() or "phase_5" in f.lower())]
    if len(hits) != 1:
        raise RuntimeError(f"expected exactly one {basename} for Phase 5, got {hits}. Set the path by hand.")
    return hits[0]


def _text(v):
    return " ".join(str(s) for s in v) if isinstance(v, list) else str(v)


def read_split(filename):
    path = hf_hub_download(REPO_ID, find_repo_file(filename), repo_type="dataset", token=HF_TOKEN)
    raw = json.load(open(path, encoding="utf-8"))
    if isinstance(raw, dict):                       # {id: {text, labels}}
        raw = [{**v, "id": k} for k, v in raw.items()]
    lk = next((k for k in _LABEL_KEYS if k in raw[0]), None)
    print(f"{filename}: {len(raw)} docs | fields {list(raw[0].keys())} | label field {lk!r}")
    return [dict(id=r["id"], text=_text(r["text"]), labels=list(dict.fromkeys(r.get(lk) or [])) if lk else [])
            for r in raw]


train_recs, dev_recs, test_recs = read_split("train.json"), read_split("dev.json"), read_split("test.json")

all_labels = sorted({l for r in train_recs + dev_recs for l in r["labels"]}, key=lambda x: (str(type(x)), x))
lab2idx = {l: i for i, l in enumerate(all_labels)}
idx2lab = {i: l for l, i in lab2idx.items()}
L = len(all_labels)


def multi_hot(recs):
    y = np.zeros((len(recs), L), dtype=np.float32)
    for i, r in enumerate(recs):
        for l in r["labels"]:
            y[i, lab2idx[l]] = 1.0
    return y


y_train, y_dev = multi_hot(train_recs), multi_hot(dev_recs)
freq = y_train.sum(0)
print(f"{L} statutes | labels/doc: train {y_train.sum(1).mean():.2f} dev {y_dev.sum(1).mean():.2f}")
print(f"train docs per statute: min {freq.min():.0f}  median {np.median(freq):.0f}  max {freq.max():.0f}")
print(f"statutes with <=5 train docs: {(freq <= 5).sum()}   in dev but never in train: {((y_dev.sum(0) > 0) & (freq == 0)).sum()}")


# ---- tokenise once (no special tokens; <s>/</s> are added per chunk), cached on Drive ----
def encode_docs(recs, tag, bs=2000, workers=None):
    workers = workers or min(4, os.cpu_count() or 1)
    path = os.path.join(p5_dir, f"tok_txl_{tag}.pkl")
    if os.path.exists(path):
        return pickle.load(open(path, "rb"))
    out = []
    for i in range(0, len(recs), bs):
        texts = [normalize_text(r["text"]) for r in recs[i:i + bs]]
        out += [np.asarray(x, dtype=np.int32) for x in
                tokenizer.encode_batch(texts, add_special_tokens=False, num_workers=workers, chunk_size=32)]
        print(f"  tokenised {min(i + bs, len(recs)):,}/{len(recs):,} ({tag})")
    pickle.dump(out, open(path, "wb"))
    return out


enc_train, enc_dev, enc_test = encode_docs(train_recs, "train"), encode_docs(dev_recs, "dev"), encode_docs(test_recs, "test")
lens = np.array([len(x) for x in enc_train])
print("train tokens/doc  median %d  p90 %d  p99 %d  max %d" % tuple(np.percentile(lens, [50, 90, 99, 100]).astype(int)))
print("-> 510-token chunks/doc  median %.1f  p90 %.1f" % tuple(np.percentile(np.ceil(lens / 510), [50, 90])))

Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

train.json: 42750 docs | fields ['id', 'text', 'labels'] | label field 'labels'


Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

dev.json: 10181 docs | fields ['id', 'text', 'labels'] | label field 'labels'


Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

test.json: 13019 docs | fields ['id', 'text'] | label field None
100 statutes | labels/doc: train 3.70 dev 3.92
train docs per statute: min 272  median 679  max 11730
statutes with <=5 train docs: 0   in dev but never in train: 0
train tokens/doc  median 635  p90 4150  p99 12304  max 140077
-> 510-token chunks/doc  median 2.0  p90 9.0


## 5. Chunking, metrics, thresholds (pure numpy; identical to the encoder–decoder notebook)

In [6]:
# ============================================================
# Pure-numpy utilities: chunking, metrics, thresholds  (identical to the GQLA notebook, except chunk_ids
# is given the BODY length 511: one <eos> is appended per chunk later, so every window is <= 512 tokens)
# ============================================================
import numpy as np


# ---------------- long-document chunking ----------------
def chunk_ids(ids, body_len=511, stride=511, min_tail=16):
    """Split one tokenised document into windows of <= body_len tokens (an <eos> is appended later).
    stride == body_len -> non-overlapping windows; stride < body_len -> overlapping windows."""
    if len(ids) <= body_len:
        return [ids]
    chunks, s = [], 0
    while True:
        chunks.append(ids[s:s + body_len])
        if s + body_len >= len(ids):
            break
        s += stride
    # a tiny trailing fragment carries almost no information
    if len(chunks) > 1 and len(chunks[-1]) < min_tail:
        chunks.pop()
    return chunks


def select_chunks(chunks, max_chunks, strategy="head_tail"):
    """Long-document representation strategy (one of the ablations)."""
    n = len(chunks)
    if n <= max_chunks:
        return chunks
    if strategy == "head":
        return chunks[:max_chunks]
    if strategy == "tail":
        return chunks[-max_chunks:]
    if strategy == "head_tail":
        t = max_chunks // 2
        return chunks[:max_chunks - t] + (chunks[-t:] if t else [])
    if strategy == "uniform":
        idx = np.unique(np.linspace(0, n - 1, max_chunks).round().astype(int))
        return [chunks[i] for i in idx]
    raise ValueError(f"unknown strategy {strategy!r}")


# ---------------- metrics ----------------
def _counts(y, pred):
    tp = (y * pred).sum(0)
    fp = ((1 - y) * pred).sum(0)
    fn = (y * (1 - pred)).sum(0)
    return tp, fp, fn


def f1_per_class(y, pred):
    tp, fp, fn = _counts(y, pred)
    den = 2 * tp + fp + fn
    return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0), y.sum(0)


def compute_metrics(y, pred):
    """macro_f1      : mean F1 over statutes that occur in `y` (primary metric)
       macro_f1_all  : mean F1 over every column, absent statutes count as 0
       micro_f1, subset_acc : the two secondary metrics"""
    f1, sup = f1_per_class(y, pred)
    tp, fp, fn = _counts(y, pred)
    return dict(
        macro_f1=float(f1[sup > 0].mean()) if (sup > 0).any() else 0.0,
        macro_f1_all=float(f1.mean()),
        micro_f1=float(2 * tp.sum() / max(2 * tp.sum() + fp.sum() + fn.sum(), 1)),
        subset_acc=float((y == pred).all(1).mean()),
    )


# ---------------- thresholding ----------------
THR_GRID = np.round(np.arange(0.05, 0.951, 0.025), 4)


def predict_labels(probs, thr, force_top1=True):
    """thr: scalar or per-class array [L]. Every document has >=1 statute, so if
    nothing clears the threshold we fall back to the arg-max statute."""
    pred = (probs >= thr).astype(np.float32)
    if force_top1:
        empty = np.where(pred.sum(1) == 0)[0]
        if len(empty):
            pred[empty, probs[empty].argmax(1)] = 1.0
    return pred


def tune_global_threshold(probs, y, grid=THR_GRID, force_top1=True):
    scores = [compute_metrics(y, predict_labels(probs, t, force_top1))["macro_f1"] for t in grid]
    i = int(np.argmax(scores))
    return float(grid[i]), float(scores[i])


def tune_per_class_threshold(probs, y, global_thr, grid=THR_GRID, min_support=3, shrink=0.5):
    """Per-statute thresholds. Statutes with < min_support positives in the tuning
    set keep the global threshold (too little evidence to tune); the rest are
    shrunk towards it:  thr_c = shrink*best_c + (1-shrink)*global."""
    n, L = probs.shape
    thr = np.full(L, global_thr, dtype=np.float32)
    sup = y.sum(0)
    for c in range(L):
        if sup[c] < min_support:
            continue
        P = probs[:, c:c + 1] >= grid[None, :]
        yc = y[:, c:c + 1] > 0
        tp = (P & yc).sum(0)
        fp = (P & ~yc).sum(0)
        fn = (~P & yc).sum(0)
        f1 = 2 * tp / np.maximum(2 * tp + fp + fn, 1)
        best = grid[np.flatnonzero(f1 == f1.max())]           # ties -> closest to global
        best = best[np.argmin(np.abs(best - global_thr))]
        thr[c] = shrink * best + (1 - shrink) * global_thr
    return thr


def honest_threshold_eval(probs, y, min_support=2, shrink=0.5, seed=0):
    """Tuning thresholds on dev and scoring on the same dev is optimistic. This
    tunes on one random half of dev and scores on the other half (and swaps)."""
    rng = np.random.RandomState(seed)
    perm = rng.permutation(len(y))
    A, B = perm[: len(y) // 2], perm[len(y) // 2:]
    res = {"fixed_0.5": [], "global": [], "per_class": []}
    for tr, te in ((A, B), (B, A)):
        g, _ = tune_global_threshold(probs[tr], y[tr])
        pc = tune_per_class_threshold(probs[tr], y[tr], g, min_support=min_support, shrink=shrink)
        res["fixed_0.5"].append(compute_metrics(y[te], predict_labels(probs[te], 0.5))["macro_f1"])
        res["global"].append(compute_metrics(y[te], predict_labels(probs[te], g))["macro_f1"])
        res["per_class"].append(compute_metrics(y[te], predict_labels(probs[te], pc))["macro_f1"])
    return {k: float(np.mean(v)) for k, v in res.items()}


def rare_report(y_dev, pred, y_train, edges=((0, 0), (1, 5), (6, 20), (21, 100), (101, 10**9))):
    """Mean F1 on dev, bucketed by how many TRAIN documents carry the statute."""
    f1, sup = f1_per_class(y_dev, pred)
    tf = y_train.sum(0)
    rows = []
    for lo, hi in edges:
        m = (tf >= lo) & (tf <= hi) & (sup > 0)
        rows.append((f"{lo}-{hi if hi < 10**9 else 'inf'}", int(m.sum()),
                     float(f1[m].mean()) if m.any() else float("nan")))
    return rows

## 6. Adaptation methods for Transformer-XL
The linear layers of a block are `attn.q_proj` (768→768), `attn.k_proj`/`attn.v_proj` (768→192, GQA with 8 KV heads), `attn.out_proj`, and the FFN layers
`ffn.0` (768→3072) and `ffn.3` (3072→768). The FFN layers have biases. `attn.r_proj` (the relative-position projection) and `u`/`v` always stay frozen.

In [7]:
# ============================================================
#   method       technique (reference)                              what is trained
#   lorma_plus   LoRMA+  (Bihany, Patel, Modi, ACL Findings 2025)    A, B of (I + a/r BA) W0  [multiplicative]
#   lorma_pi     LoRMA-pi (same paper, permutation inflation)       A, B of I_pi(BA) W0
#   adalora      AdaLoRA (Zhang et al., 2023)                       P, E, Q (SVD form) + importance-based rank pruning
#   qlora        QLoRA   (Dettmers et al., 2023)                    LoRA on top of a frozen 4-bit NF4 backbone
#   lora         LoRA    (Hu et al., 2022)                          A, B of W0 + a/r BA   (extra)
#   adapter      Houlsby adapters (2019)                            bottleneck MLPs after attention and after the FFN
#   prefix       Prefix tuning (Li & Liang, 2021)                   learned Transformer-XL memory per block
#   prompt       Prompt tuning (Lester et al., 2021)                soft tokens before every chunk
#   partial      Partial / layer-wise fine-tuning                   top-k blocks + final norm
#   full         Full fine-tuning                                   everything incl. the embedding
# ============================================================
class LoRALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base = base
        for p in base.parameters():
            p.requires_grad = False
        self.in_features, self.out_features = base.in_features, base.out_features
        self.A = nn.Parameter(torch.empty(r, self.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(self.out_features, r))
        self.scale, self.drop = alpha / r, nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A.to(x.dtype)), self.B.to(x.dtype))


class LoRMALinear(nn.Module):
    """'plus': W = (I + (a/r) B A) W0, B=0 -> identity; evaluated as y0 + s*B(A y0), y0 = W0 x (no d x d matrix).
       'pi'  : W = I_pi(BA) W0, row i of BA rotated cyclically by i (rank inflation); init B[:,0]=1, A[0,0]=1 -> I_pi(BA)=I.
       The bias (FFN layers) is added after the multiplicative transform: it is not part of W0."""

    def __init__(self, base, r=8, alpha=16, variant="plus", dropout=0.0):
        super().__init__()
        self.base = base
        for p in base.parameters():
            p.requires_grad = False
        d = base.out_features
        self.in_features, self.out_features = base.in_features, d
        self.variant, self.scale, self.drop = variant, alpha / r, nn.Dropout(dropout)
        self.A, self.B = nn.Parameter(torch.empty(r, d)), nn.Parameter(torch.empty(d, r))
        if variant == "plus":
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5)); nn.init.zeros_(self.B)
        else:
            nn.init.zeros_(self.A); self.A.data[0, 0] = 1.0
            nn.init.normal_(self.B, std=0.02); self.B.data[:, 0] = 1.0
            self.register_buffer("rot_idx", (torch.arange(d)[None, :] - torch.arange(d)[:, None]) % d, persistent=False)

    def forward(self, x):
        y0 = F.linear(x, self.base.weight.to(x.dtype))
        if self.variant == "plus":
            y = y0 + self.scale * F.linear(F.linear(self.drop(y0), self.A.to(y0.dtype)), self.B.to(y0.dtype))
        else:
            M = torch.gather(self.B @ self.A, 1, self.rot_idx)
            y = F.linear(y0, M.to(y0.dtype))
        return y if self.base.bias is None else y + self.base.bias.to(y.dtype)


class AdaLoRALinear(nn.Module):
    """W0 x (+ b) + a/r_init * P diag(E) Q x ; E starts at 0 -> identity. E is pruned by AdaLoRAAllocator."""

    def __init__(self, base, r=12, alpha=32):
        super().__init__()
        self.base = base
        for p in base.parameters():
            p.requires_grad = False
        self.in_features, self.out_features = base.in_features, base.out_features
        self.scale = alpha / r
        self.P = nn.Parameter(torch.randn(base.out_features, r) * 0.02)
        self.Q = nn.Parameter(torch.randn(r, base.in_features) * 0.02)
        self.E = nn.Parameter(torch.zeros(r))

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(x, self.Q.to(x.dtype)) * self.E.to(x.dtype), self.P.to(x.dtype))


NF4_CODE = torch.tensor([-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453,
                         -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0,
                         0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224,
                         0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0])


class NF4Linear(nn.Module):
    """Frozen linear stored as 4-bit NormalFloat codes (2 per byte) + one fp16 absmax per block of 64, de-quantised in forward."""

    def __init__(self, base, block=64):
        super().__init__()
        W = base.weight.detach().float()
        self.shape, self.block = tuple(W.shape), block
        flat = W.reshape(-1)
        pad = (-flat.numel()) % block
        flat = torch.cat([flat, flat.new_zeros(pad)]).view(-1, block)
        absmax = flat.abs().amax(1, keepdim=True).clamp_min(1e-8)
        code = NF4_CODE.to(flat.device)
        idx = torch.empty(flat.shape, dtype=torch.uint8, device=flat.device)
        for i in range(0, flat.size(0), 4096):
            idx[i:i + 4096] = ((flat[i:i + 4096] / absmax[i:i + 4096]).unsqueeze(-1) - code).abs().argmin(-1)
        idx = idx.view(-1)
        if idx.numel() % 2:
            idx = torch.cat([idx, idx.new_zeros(1)])
        self.register_buffer("packed", (idx[0::2] << 4) | idx[1::2])
        self.register_buffer("absmax", absmax.half())
        self.register_buffer("code", code.clone())
        self.n = W.numel()
        self.bias = None if base.bias is None else nn.Parameter(base.bias.detach().clone(), requires_grad=False)
        self.in_features, self.out_features = self.shape[1], self.shape[0]

    def dequant(self, dtype):
        idx = torch.stack([self.packed >> 4, self.packed & 15], 1).view(-1).long()
        w = self.code[idx][: self.absmax.numel() * self.block].view(-1, self.block) * self.absmax.float()
        return w.view(-1)[: self.n].view(self.shape).to(dtype)

    def forward(self, x):
        return F.linear(x, self.dequant(x.dtype), None if self.bias is None else self.bias.to(x.dtype))


class Adapter(nn.Module):
    def __init__(self, d, bottleneck=64, dropout=0.1):
        super().__init__()
        self.down, self.up = nn.Linear(d, bottleneck), nn.Linear(bottleneck, d)
        self.drop = nn.Dropout(dropout)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return x + self.drop(self.up(F.gelu(self.down(x))))


class AdapterAfter(nn.Module):
    """adapter on the OUTPUT of a sub-layer, before the residual (Houlsby placement)"""

    def __init__(self, inner, d, bottleneck=64, dropout=0.1):
        super().__init__()
        self.inner, self.adapter = inner, Adapter(d, bottleneck, dropout)

    def forward(self, *a, **k):
        return self.adapter(self.inner(*a, **k))


class AdaLoRAAllocator:
    """Importance-based rank allocation (Zhang et al., 2023): smoothed sensitivity |w * grad| x uncertainty, triplet score
    per singular value, cubic budget schedule from b0 to target_r * n_modules, frozen pattern in the last t_final share
    of steps, plus the orthogonality regulariser on P and Q. (Same implementation as the encoder-decoder notebook.)"""

    def __init__(self, mods, target_r, t_init, t_final, delta_t, beta, orth):
        self.mods, self.delta_t, self.beta, self.orth = mods, delta_t, beta, orth
        self.t_init_frac, self.t_final_frac = t_init, t_final
        self.b0 = sum(m.E.numel() for m in mods)
        self.bT = int(target_r * len(mods))
        self.ipt, self.unc, self.keep, self.total = {}, {}, None, None

    def setup(self, total_steps):
        self.total = total_steps
        self.ti, self.tf = int(self.t_init_frac * total_steps), int(self.t_final_frac * total_steps)
        self.keep = None

    def orth_loss(self):
        loss = 0.0
        for m in self.mods:
            I = torch.eye(m.P.size(1), device=m.P.device)
            loss = loss + torch.norm(m.P.t() @ m.P - I) + torch.norm(m.Q @ m.Q.t() - I)
        return self.orth * loss / len(self.mods)

    @torch.no_grad()
    def update_importance(self):
        b = self.beta
        for m in self.mods:
            for p in (m.P, m.E, m.Q):
                if p.grad is None:
                    continue
                s = torch.nan_to_num((p * p.grad).abs(), nan=0.0, posinf=0.0, neginf=0.0)
                k = id(p)
                if k not in self.ipt:
                    self.ipt[k], self.unc[k] = torch.zeros_like(p), torch.zeros_like(p)
                self.ipt[k].mul_(b).add_(s, alpha=1 - b)
                self.unc[k].mul_(b).add_((s - self.ipt[k]).abs(), alpha=1 - b)

    @torch.no_grad()
    def _scores(self):
        z = lambda p: self.ipt.get(id(p), torch.zeros_like(p)) * self.unc.get(id(p), torch.zeros_like(p))
        return [z(m.E) + z(m.P).mean(0) + z(m.Q).mean(1) for m in self.mods]

    @torch.no_grad()
    def _prune_to(self, budget):
        scores = self._scores()
        allv = torch.cat(scores)
        if budget >= allv.numel():
            return [torch.ones_like(s, dtype=torch.bool) for s in scores]
        thr = torch.topk(allv, budget).values[-1]
        keep = [s >= thr for s in scores]
        for m, kp in zip(self.mods, keep):
            m.E.data.mul_(kp.to(m.E.dtype))
        return keep

    def budget_at(self, step):
        mul = 1 - (step - self.ti) / max(1, self.total - self.tf - self.ti)
        return int(self.bT + (self.b0 - self.bT) * max(mul, 0.0) ** 3)

    @torch.no_grad()
    def after_step(self, step):
        if self.total is None or step <= self.ti:
            return
        if step > self.total - self.tf:
            if self.keep is None:
                self.keep = self._prune_to(self.bT)
            for m, kp in zip(self.mods, self.keep):
                m.E.data.mul_(kp.to(m.E.dtype))
        elif step % self.delta_t == 0:
            self._prune_to(self.budget_at(step))

    def effective_ranks(self):
        return [int((m.E != 0).sum()) for m in self.mods]


class PEFTHandle:
    def __init__(self, method, spec):
        self.method, self.spec, self.params, self.mods, self.allocator = method, spec, [], [], None

    def param_ids(self): return {id(p) for p in self.params}
    def setup(self, total):
        if self.allocator is not None: self.allocator.setup(total)
    def extra_loss(self): return self.allocator.orth_loss() if self.allocator is not None else 0.0
    def after_backward(self, step):
        if self.allocator is not None: self.allocator.update_importance()
    def after_step(self, step):
        if self.allocator is not None: self.allocator.after_step(step)

    def describe(self):
        n = sum(p.numel() for p in self.params)
        extra = ""
        if self.allocator is not None:
            r = self.allocator.effective_ranks()
            extra = f" | AdaLoRA ranks now min {min(r)} / mean {sum(r)/len(r):.1f} / max {max(r)}"
        return f"PEFT '{self.method}': {n/1e6:.3f}M new parameters in {len(self.mods)} modules{extra}"


TXL_LINEARS = ("q_proj", "k_proj", "v_proj", "out_proj", "ffn0", "ffn3")
PEFT_DEFAULTS = {
    "lorma_plus": dict(r=8, alpha=16, top_layers=4, targets=("q_proj", "v_proj")),
    "lorma_pi":   dict(r=8, top_layers=4, targets=("q_proj", "v_proj")),
    "lora":       dict(r=8, alpha=16, top_layers=4, targets=("q_proj", "v_proj")),
    "adalora":    dict(init_r=12, target_r=6, alpha=32, top_layers=4, targets=TXL_LINEARS,
                       t_init=0.1, t_final=0.2, delta_t=10, beta=0.85, orth=0.1),
    "qlora":      dict(r=16, alpha=32, top_layers=4, targets=TXL_LINEARS, quant_all_layers=True),
    "adapter":    dict(bottleneck=64, top_layers=4),
    "prefix":     dict(n_tokens=16),
    "prompt":     dict(n_tokens=20),
    "partial":    dict(k=3),
    "full":       dict(),
}


def _get_lin(blk, name):
    return blk.ffn[int(name[3:])] if name.startswith("ffn") else getattr(blk.attn, name)


def _set_lin(blk, name, mod):
    if name.startswith("ffn"):
        blk.ffn[int(name[3:])] = mod
    else:
        setattr(blk.attn, name, mod)


@torch.no_grad()
def real_memory(enc, P):
    """layer inputs at P positions of a real training chunk -> prefix-memory init at the right scale [n_blocks, P, d]"""
    doc = next(d for d in enc_train if len(d) >= 300)
    ids = torch.as_tensor(np.concatenate([[BOS_ID], doc[:510], [EOS_ID]]), dtype=torch.long, device=device)[None]
    _, ins = txl_hidden(enc, ids, return_layer_inputs=True)
    pos = torch.randperm(ids.size(1), device=device)[:P].sort().values
    return torch.stack([h[0, pos] for h in ins]).float()


def apply_peft(bb, spec):
    """inject spec = dict(method=..., **overrides) into a TXLBackbone whose Phase-1 weights are frozen -> PEFTHandle"""
    spec = dict(spec); method = spec.pop("method")
    c = {**PEFT_DEFAULTS[method], **spec}
    h = PEFTHandle(method, c)
    enc = bb.enc
    n = len(enc.blocks)
    first = max(0, n - c["top_layers"]) if "top_layers" in c else 0
    if method == "partial":
        for blk in enc.blocks[max(0, n - c["k"]):]:
            for p in blk.parameters(): p.requires_grad = True
        if enc.out_norm == "final":
            for p in enc.final_norm.parameters(): p.requires_grad = True
        for blk in enc.blocks:                                    # positional pieces stay frozen
            blk.attn.r_proj.weight.requires_grad = False
    elif method == "full":
        for p in enc.parameters(): p.requires_grad = True
    elif method == "prompt":
        emb = enc.embed.weight
        bb.soft_prompt = nn.Parameter(emb.data[torch.randint(1000, min(20000, emb.size(0)), (c["n_tokens"],))].clone().float())
        h.params.append(bb.soft_prompt)
    elif method == "prefix":
        enc.prefix = nn.Parameter(real_memory(enc, c["n_tokens"]))
        h.params.append(enc.prefix)
    elif method == "adapter":
        for blk in enc.blocks[first:]:
            blk.attn = AdapterAfter(blk.attn, D_MODEL, c["bottleneck"]).to(device)
            blk.ffn = AdapterAfter(blk.ffn, D_MODEL, c["bottleneck"]).to(device)
            for m in (blk.attn.adapter, blk.ffn.adapter):
                h.mods.append(m); h.params += list(m.parameters())
    else:
        lo = 0 if (method == "qlora" and c["quant_all_layers"]) else first
        for i in range(lo, n):
            blk, adapted = enc.blocks[i], i >= first
            for name in c["targets"]:
                base = _get_lin(blk, name)
                if method in ("lorma_plus", "lorma_pi"):
                    new = LoRMALinear(base, c["r"], c.get("alpha", 16), "plus" if method == "lorma_plus" else "pi")
                    ps = [new.A, new.B]
                elif method == "lora":
                    new = LoRALinear(base, c["r"], c["alpha"]); ps = [new.A, new.B]
                elif method == "adalora":
                    new = AdaLoRALinear(base, c["init_r"], c["alpha"]); ps = [new.P, new.Q, new.E]
                elif method == "qlora":
                    q = NF4Linear(base)
                    new, ps = (LoRALinear(q, c["r"], c["alpha"]), None) if adapted else (q, [])
                    if adapted: ps = [new.A, new.B]
                else:
                    raise ValueError(method)
                _set_lin(blk, name, new.to(device))
                if ps:
                    h.mods.append(new); h.params += ps
        if method == "adalora":
            h.allocator = AdaLoRAAllocator(h.mods, c["target_r"], c["t_init"], c["t_final"], c["delta_t"], c["beta"], c["orth"])
    if method in ("qlora",):
        gc.collect(); torch.cuda.empty_cache()
    return h


def freeze(m):
    for p in m.parameters():
        p.requires_grad = False

## 7. Model: Transformer-XL → chunk vectors → chunk-level BiGRU / BiLSTM / Transformer → statute head
`mode="cached"` runs the frozen backbone **once** and stores `[mean | max | last]` per chunk in fp16, so only the head is trained (seconds per epoch, good for ablations).
`mode="token"` runs the backbone every step and adapts it with `peft=dict(method=...)`.

In [8]:
PART_NAMES = ("mean", "max", "last")
CHUNK_BODY = 510


class TXLBackbone(nn.Module):
    """Owns a deep copy of the Phase-1 Transformer-XL (optionally truncated to n_blocks). All Phase-1 weights frozen at start.
    The Transformer-XL itself always stays in eval mode (no dropout), as in the encoder-decoder notebook."""

    def __init__(self, n_blocks=None, grad_ckpt=True):
        super().__init__()
        self.enc = copy.deepcopy(TXL_BASE).truncate(n_blocks).to(device)
        freeze(self.enc)
        self.soft_prompt, self.grad_ckpt = None, grad_ckpt

    def train(self, mode=True):
        super().train(mode); self.enc.eval(); return self

    def forward(self, ids, tok_mask):
        ck = self.grad_ckpt and self.training and torch.is_grad_enabled()
        if self.soft_prompt is None:
            return txl_hidden(self.enc, ids, ckpt=ck)
        e = self.enc.embed(ids)
        P = self.soft_prompt.size(0)
        x = torch.cat([self.soft_prompt.to(e.dtype).unsqueeze(0).expand(e.size(0), -1, -1), e], 1)
        return txl_hidden(self.enc, embeds=x, ckpt=ck)[:, P:]


def pool_parts(H, tok_mask):
    """mean / max over real tokens, and the LAST real token (</s>): in a causal model the only state that has read the whole chunk"""
    H = H.float()
    m = tok_mask.unsqueeze(-1)
    ln = tok_mask.sum(1)
    return {"mean": (H * m).sum(1) / ln[:, None],
            "max": H.masked_fill(~m, -1e4).max(1).values,
            "last": H[torch.arange(H.size(0), device=H.device), ln - 1]}


def parse_parts(s):
    parts = tuple(p for p in PART_NAMES if p in s.split("+"))
    bad = set(s.split("+")) - set(PART_NAMES)
    if bad or not parts:
        raise ValueError(f"parts={s!r}: use a '+'-joined subset of {PART_NAMES}")
    return parts


def _pack_run(rnn, x, lengths):
    packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
    out, _ = rnn(packed)
    out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
    return out


class ChunkSeqEncoder(nn.Module):
    """'gru' | 'lstm' (bidirectional RNN) | 'transformer' (small BERT-style, learned chunk positions) | 'none'"""

    def __init__(self, d_in, hid=256, kind="gru", layers=1, dropout=0.3, max_chunks=64, heads=4):
        super().__init__()
        self.kind, D = kind, 2 * hid
        self.inp = nn.Sequential(nn.LayerNorm(d_in), nn.Dropout(dropout), nn.Linear(d_in, D), nn.GELU())
        if kind in ("gru", "lstm"):
            cls = {"gru": nn.GRU, "lstm": nn.LSTM}[kind]
            self.rnn = cls(D, hid, layers, batch_first=True, bidirectional=True, dropout=dropout if layers > 1 else 0.0)
        elif kind == "transformer":
            self.pos = nn.Embedding(max_chunks, D)
            layer = nn.TransformerEncoderLayer(D, heads, 4 * D, dropout, activation="gelu", batch_first=True, norm_first=True)
            self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        elif kind != "none":
            raise ValueError(kind)
        self.out_dim = D

    def forward(self, x, chunk_mask):
        z = self.inp(x)
        if self.kind in ("gru", "lstm"):
            z = _pack_run(self.rnn, z, chunk_mask.sum(1))
        elif self.kind == "transformer":
            z = self.enc(z + self.pos(torch.arange(z.size(1), device=z.device)), src_key_padding_mask=~chunk_mask)
        return z


class DocHead(nn.Module):
    """'attn' (one attention over chunks) | 'label_attn' (one attention per statute) | 'meanmax'"""

    def __init__(self, dim, n_labels, pool="attn", dropout=0.3):
        super().__init__()
        self.pool, self.drop, self.last_attn = pool, nn.Dropout(dropout), None
        if pool == "attn":
            self.att = nn.Sequential(nn.Linear(dim, dim // 2), nn.Tanh(), nn.Linear(dim // 2, 1))
            self.fc = nn.Linear(dim, n_labels)
        elif pool == "label_attn":
            self.Q = nn.Parameter(torch.empty(n_labels, dim)); nn.init.xavier_uniform_(self.Q)
            self.W = nn.Parameter(torch.empty(n_labels, dim)); nn.init.xavier_uniform_(self.W)
            self.b = nn.Parameter(torch.zeros(n_labels))
        elif pool == "meanmax":
            self.fc = nn.Linear(2 * dim, n_labels)
        else:
            raise ValueError(pool)

    def forward(self, z, mask):
        if self.pool == "attn":
            a = torch.softmax(self.att(z).squeeze(-1).masked_fill(~mask, -1e4), -1)
            self.last_attn = a.detach()
            return self.fc(self.drop((a.unsqueeze(-1) * z).sum(1)))
        if self.pool == "label_attn":
            a = torch.softmax(torch.einsum("bnd,ld->bln", z, self.Q).masked_fill(~mask[:, None, :], -1e4), -1)
            self.last_attn = a.detach()
            return (self.drop(torch.einsum("bln,bnd->bld", a, z)) * self.W).sum(-1) + self.b
        m = mask.unsqueeze(-1)
        mean = (z * m).sum(1) / m.sum(1)
        return self.fc(self.drop(torch.cat([mean, z.masked_fill(~m, -1e4).max(1).values], -1)))


class TXLTokenClassifier(nn.Module):
    """mode 'token': backbone on the fly (adapted by cfg.peft), micro-batches of chunks -> chunk encoder -> statute head"""

    def __init__(self, n_labels, cfg):
        super().__init__()
        self.parts = parse_parts(cfg.parts)
        self.backbone = TXLBackbone(cfg.n_blocks)
        self.peft = apply_peft(self.backbone, cfg.peft) if cfg.peft else None
        self.chunk = ChunkSeqEncoder(len(self.parts) * D_MODEL, cfg.chunk_hid, cfg.chunk_rnn, cfg.chunk_layers, cfg.dropout, cfg.max_chunks)
        self.head = DocHead(self.chunk.out_dim, n_labels, cfg.pool, cfg.dropout)
        self.chunk_bs = cfg.txl_chunk_bs

    def forward(self, ids, tok_mask, chunk_mask):                     # [B,N,T], [B,N,T], [B,N]
        B, N, T = ids.shape
        flat_ids, flat_tm = ids[chunk_mask], tok_mask[chunk_mask]
        vecs = []
        for i in range(0, flat_ids.size(0), self.chunk_bs):
            tm = flat_tm[i:i + self.chunk_bs]
            t = int(tm.sum(1).max())
            p = pool_parts(self.backbone(flat_ids[i:i + self.chunk_bs, :t], tm[:, :t]), tm[:, :t])
            vecs.append(torch.cat([p[k] for k in self.parts], -1))
        vecs = torch.cat(vecs)
        x = vecs.new_zeros(B, N, vecs.size(-1))
        x[chunk_mask] = vecs
        return self.head(self.chunk(x, chunk_mask), chunk_mask)


class CachedFeatureClassifier(nn.Module):
    def __init__(self, in_dim, n_labels, cfg):
        super().__init__()
        self.chunk = ChunkSeqEncoder(in_dim, cfg.chunk_hid, cfg.chunk_rnn, cfg.chunk_layers, cfg.dropout, cfg.max_chunks)
        self.head = DocHead(self.chunk.out_dim, n_labels, cfg.pool, cfg.dropout)

    def forward(self, feats, chunk_mask):
        return self.head(self.chunk(feats, chunk_mask), chunk_mask)


def backbone_footprint(n_blocks=None, enc=None):
    """(params, bytes) of what a deployed pipeline needs from the backbone: embedding + used blocks (+ final norm); NF4 counted packed"""
    if enc is None:
        n = N_BLOCKS if n_blocks is None else n_blocks
        mods = [TXL_BASE.embed] + list(TXL_BASE.blocks[:n]) + ([TXL_BASE.final_norm] if n == N_BLOCKS else [])
    else:
        mods = [enc.embed] + list(enc.blocks) + ([enc.final_norm] if enc.out_norm == "final" else [])
    params = nbytes = 0
    for m in mods:
        for p in m.parameters():
            params += p.numel(); nbytes += p.numel() * p.element_size()
        for q in m.modules():
            if isinstance(q, NF4Linear):
                params += q.n; nbytes += q.packed.numel() + q.absmax.numel() * 2
    return params, nbytes


def model_footprint(model, mode, n_blocks=None):
    if mode == "token":
        p0, b0 = backbone_footprint(enc=model.backbone.enc)
        extra = [model.backbone.soft_prompt] if model.backbone.soft_prompt is not None else []
        if model.backbone.enc.prefix is not None:
            extra.append(model.backbone.enc.prefix)
        head = [p for n, p in model.named_parameters() if not n.startswith("backbone.")] + extra
    else:
        p0, b0 = backbone_footprint(n_blocks)
        head = list(model.parameters())
    return (p0 + sum(p.numel() for p in head)) / 1e6, (b0 + sum(p.numel() * p.element_size() for p in head)) / 1e6

## 8. Datasets, collate, fast feature extraction (token-budget batching over length-sorted chunks)

In [9]:
def amp_context(enabled=True):
    return torch.autocast(device.type, dtype=AMP_DTYPE or torch.bfloat16, enabled=bool(enabled and AMP_DTYPE is not None))


class ChunkedDocs(Dataset):
    """each document -> list of <=510-token chunk bodies (after the long-document strategy); <s>/</s> added in collate"""

    def __init__(self, enc_docs, y, cfg):
        self.items = [select_chunks(chunk_ids(d, CHUNK_BODY, cfg.chunk_stride), cfg.max_chunks, cfg.chunk_strategy) for d in enc_docs]
        self.y = y

    def __len__(self): return len(self.items)
    def __getitem__(self, i): return i, self.items[i], (None if self.y is None else self.y[i])


class FeatDocs(Dataset):
    def __init__(self, feats, y): self.feats, self.y = feats, y
    def __len__(self): return len(self.feats)
    def __getitem__(self, i): return i, self.feats[i], (None if self.y is None else self.y[i])


def _stack_y(batch):
    return None if batch[0][2] is None else torch.from_numpy(np.stack([b[2] for b in batch]))


def collate_tokens(batch):
    B = len(batch)
    N = max(len(c) for _, c, _ in batch)
    T = max(len(x) for _, c, _ in batch for x in c) + 2                     # <s> ... </s>
    ids = torch.full((B, N, T), PAD_ID, dtype=torch.long)
    tm = torch.zeros(B, N, T, dtype=torch.bool)
    cm = torch.zeros(B, N, dtype=torch.bool)
    for b, (_, chunks, _) in enumerate(batch):
        for n, c in enumerate(chunks):
            k = len(c)
            ids[b, n, 0] = BOS_ID
            ids[b, n, 1:k + 1] = torch.from_numpy(np.asarray(c, dtype=np.int64))
            ids[b, n, k + 1] = EOS_ID
            tm[b, n, :k + 2] = True
            cm[b, n] = True
    return dict(ids=ids, tok_mask=tm, chunk_mask=cm, y=_stack_y(batch), idx=torch.tensor([b[0] for b in batch]))


def feat_columns(parts, d):
    return [i * d + j for i, name in enumerate(PART_NAMES) if name in parts for j in range(d)]


def make_collate_feats(cols):
    cols = torch.as_tensor(cols, dtype=torch.long)

    def collate_feats(batch):
        B, N = len(batch), max(f.size(0) for _, f, _ in batch)
        x = torch.zeros(B, N, len(cols))
        cm = torch.zeros(B, N, dtype=torch.bool)
        for b, (_, f, _) in enumerate(batch):
            x[b, :f.size(0)] = f.index_select(1, cols).float()
            cm[b, :f.size(0)] = True
        return dict(feats=x, chunk_mask=cm, y=_stack_y(batch), idx=torch.tensor([b[0] for b in batch]))
    return collate_feats


@torch.no_grad()
def extract_chunk_features(ds, n_blocks=None, tok_budget=24576, max_bs=128, log_every=25):
    """frozen Transformer-XL (optionally its first n_blocks) run ONCE over every chunk -> fp16 [N_i, 3*d] = [mean|max|last]"""
    view = txl_view(n_blocks)
    TXL_BASE.to(device).eval()
    flat = [c for chunks in ds.items for c in chunks]
    doc_len = [len(chunks) for chunks in ds.items]
    n, d = len(flat), D_MODEL
    lens = np.fromiter((len(c) + 2 for c in flat), dtype=np.int64, count=n)
    order = np.argsort(lens, kind="stable")
    out = torch.empty(n, len(PART_NAMES) * d, dtype=torch.float16)
    print(f"  {n:,} chunks | {int(lens.sum()):,} tokens | blocks {n_blocks or N_BLOCKS} | amp {AMP_DTYPE} | budget {tok_budget} tok/batch")

    def run(sel):
        T = int(lens[sel].max())
        arr = np.full((len(sel), T), PAD_ID, dtype=np.int64)
        for j, i in enumerate(sel):
            k = lens[i] - 2
            arr[j, 0] = BOS_ID; arr[j, 1:k + 1] = flat[i]; arr[j, k + 1] = EOS_ID
        ids = torch.from_numpy(arr).to(device, non_blocking=True)
        tm = torch.arange(T, device=device)[None, :] < torch.from_numpy(lens[sel]).to(device)[:, None]
        with amp_context():
            H = txl_hidden(view, ids)
        p = pool_parts(H, tm)
        out[torch.from_numpy(sel)] = torch.cat([p[k] for k in PART_NAMES], -1).half().cpu()

    def safe_run(sel):
        try:
            run(sel)
        except torch.cuda.OutOfMemoryError:
            if len(sel) == 1:
                raise
            torch.cuda.empty_cache(); mid = len(sel) // 2
            safe_run(sel[:mid]); safe_run(sel[mid:])

    t0, done, nb, s = time.time(), 0, 0, 0
    while s < n:
        e = s + 1
        while e < n and (e - s) < max_bs and (e + 1 - s) * lens[order[e]] <= tok_budget:
            e += 1
        sel = order[s:e]
        safe_run(sel)
        done += int(lens[sel].sum()); s = e; nb += 1
        if nb % log_every == 0 or s >= n:
            dt = time.time() - t0
            print(f"  features {s:,}/{n:,} chunks | {done/dt:,.0f} tok/s | {dt/60:.1f} min | ETA {dt*(lens.sum()-done)/max(done,1)/60:.1f} min")
    feats, p = [], 0
    for k in doc_len:
        feats.append(out[p:p + k]); p += k
    return feats


def get_features(split, enc_docs, cfg):
    nb = cfg.n_blocks or N_BLOCKS
    name = f"txlfeat_{CKPT_SIG}_{split}_S{cfg.chunk_stride}_K{cfg.max_chunks}_{cfg.chunk_strategy}_B{nb}.pt"
    drive_path, local_path = os.path.join(p5_dir, name), os.path.join(FEAT_LOCAL, name)
    if not os.path.exists(local_path) and os.path.exists(drive_path):
        shutil.copy(drive_path, local_path)
    if os.path.exists(local_path):
        return torch.load(local_path, weights_only=False)
    print(f"extracting Transformer-XL features for '{split}' ...")
    feats = extract_chunk_features(ChunkedDocs(enc_docs, None, cfg), cfg.n_blocks)
    torch.save(feats, local_path); shutil.copy(local_path, drive_path)
    return feats

## 9. Self-tests (about a minute, run once)
1. The lean attention reproduces the Phase-1 attention, with and without XL memory. fp16 autocast is checked against fp32 and switched off if it is not finite and close.
2. Every PEFT method starts as the **identity** of the Phase-1 backbone. QLoRA (4-bit rounding), prefix and prompt tuning change the output by design, so their difference is printed rather than asserted.

In [10]:
@torch.no_grad()
def check_backbone():
    global AMP_DTYPE
    bb = TXL_BASE.to(device).eval()
    docs = [d for d in enc_train if len(d) >= 300][:4]
    ids = torch.as_tensor(np.stack([np.concatenate([[BOS_ID], d[:254], [EOS_ID]]) for d in docs]), dtype=torch.long, device=device)
    for mem in (None, torch.randn(N_BLOCKS, 16, D_MODEL, device=device)):
        bb.prefix = mem
        TransformerXLGQAAttention.forward = TransformerXLGQAAttention._orig_forward
        ref = txl_hidden(bb, ids)
        TransformerXLGQAAttention.forward = txl_attn_forward
        new = txl_hidden(bb, ids)
        d = (ref - new).abs().max().item()
        print(f"lean vs Phase-1 attention ({'16 memory slots' if mem is not None else 'no memory'}): max|diff| {d:.2e}")
        assert d < 1e-3 * max(1.0, ref.abs().max().item()), "patched Transformer-XL attention differs"
    bb.prefix = None
    if AMP_DTYPE is not None:
        with amp_context():
            h16 = txl_hidden(bb, ids)
        ok = bool(torch.isfinite(h16).all())
        cos = F.cosine_similarity(h16.float().reshape(-1, D_MODEL), new.reshape(-1, D_MODEL), -1).min().item() if ok else 0.0
        print(f"{AMP_DTYPE} autocast: finite={ok}, min cosine to fp32 {cos:.4f}")
        if not ok or cos < 0.98:
            AMP_DTYPE = None
            print("-> mixed precision switched OFF (fp32 everywhere)")


@torch.no_grad()
def check_peft_identity():
    docs = [d for d in enc_train if len(d) >= 200][:4]
    ids = torch.as_tensor(np.stack([np.concatenate([[BOS_ID], d[:198], [EOS_ID]]) for d in docs]), dtype=torch.long, device=device)
    tm = torch.ones_like(ids, dtype=torch.bool)
    ref = pool_parts(txl_hidden(TXL_BASE.to(device).eval(), ids), tm)["mean"]
    print(f"{'method':12s} {'new params':>11s} {'max|diff| vs Phase-1':>22s}")
    for meth in ("lorma_plus", "lorma_pi", "lora", "adalora", "adapter", "qlora", "prefix", "prompt"):
        bb = TXLBackbone(); h = apply_peft(bb, dict(method=meth)); bb.eval()
        d = float((pool_parts(bb(ids, tm), tm)["mean"] - ref).abs().max())
        exact = meth not in ("qlora", "prefix", "prompt")
        print(f"{meth:12s} {sum(p.numel() for p in h.params):11,d} {d:22.2e}" + (("   OK" if d < 1e-4 else "   <-- NOT identity!") if exact else ""))
        if exact:
            assert d < 1e-4, f"{meth} is not the identity at initialisation"
        del bb, h
    torch.cuda.empty_cache()


check_backbone()
check_peft_identity()

lean vs Phase-1 attention (no memory): max|diff| 9.54e-07
lean vs Phase-1 attention (16 memory slots): max|diff| 1.67e-06
torch.float16 autocast: finite=True, min cosine to fp32 0.6575
-> mixed precision switched OFF (fp32 everywhere)
method        new params   max|diff| vs Phase-1
lorma_plus        61,440               0.00e+00   OK
lorma_pi          61,440               0.00e+00   OK
lora              79,872               0.00e+00   OK
adalora          608,544               0.00e+00   OK
adapter          793,088               0.00e+00   OK
qlora            811,008               3.56e-02
prefix            98,304               5.49e-02
prompt            15,360               5.66e-02


## 10. Losses, training, evaluation (as in the encoder–decoder notebook: PEFT optimiser groups, AdaLoRA hooks, KD, cost tracking)

In [11]:
class AsymmetricLoss(nn.Module):
    """ASL (Ridnik et al., 2021): down-weights the many easy negatives of a multi-label problem."""

    def __init__(self, gamma_neg=4.0, gamma_pos=0.0, clip=0.05, eps=1e-8):
        super().__init__()
        self.gn, self.gp, self.clip, self.eps = gamma_neg, gamma_pos, clip, eps

    def forward(self, logits, y):
        p = torch.sigmoid(logits)
        xs_neg = (1 - p + self.clip).clamp(max=1) if self.clip > 0 else 1 - p
        los = y * torch.log(p.clamp(min=self.eps)) + (1 - y) * torch.log(xs_neg.clamp(min=self.eps))
        pt = p * y + xs_neg * (1 - y)
        gamma = self.gp * y + self.gn * (1 - y)
        return -(los * (1 - pt) ** gamma).sum() / logits.size(0)


def build_loss(cfg, y_tr):
    if cfg.loss == "bce":
        return nn.BCEWithLogitsLoss()
    if cfg.loss == "bce_pw":
        pos = y_tr.sum(0)
        pw = np.clip(((len(y_tr) - pos) / np.maximum(pos, 1)) ** cfg.pw_power, 1.0, cfg.pw_max)
        return nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pw, dtype=torch.float32))
    if cfg.loss == "asl":
        return AsymmetricLoss()
    raise ValueError(cfg.loss)


def forward_batch(model, batch, mode):
    cm = batch["chunk_mask"].to(device)
    if mode == "token":
        return model(batch["ids"].to(device), batch["tok_mask"].to(device), cm)
    return model(batch["feats"].to(device), cm)


def resolve_amp(cfg, mode):
    want = (mode == "token") if getattr(cfg, "amp", None) is None else bool(cfg.amp)
    return want and AMP_DTYPE is not None and device.type == "cuda"


@torch.no_grad()
def predict_logits(model, loader, mode, amp=False):
    model.eval()
    out = []
    for b in loader:
        with amp_context(amp):
            out.append(forward_batch(model, b, mode).float().cpu().numpy())
    return np.concatenate(out)


def predict_probs(model, loader, mode, amp=False):
    return 1.0 / (1.0 + np.exp(-np.clip(predict_logits(model, loader, mode, amp), -30, 30)))


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def train_model(model, mode, train_loader, dev_loader, y_dev, cfg, y_tr, tag):
    loss_fn = build_loss(cfg, y_tr).to(device)
    peft = getattr(model, "peft", None)
    peft_ids = peft.param_ids() if peft is not None else set()
    new_p = [p for n, p in model.named_parameters() if p.requires_grad and id(p) in peft_ids]
    bbp = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("backbone.") and id(p) not in peft_ids]
    rest = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("backbone.") and id(p) not in peft_ids]
    groups = ([dict(params=rest, lr=cfg.lr)] + ([dict(params=new_p, lr=cfg.lr_peft)] if new_p else [])
              + ([dict(params=bbp, lr=cfg.lr_backbone)] if bbp else []))
    opt = torch.optim.AdamW(groups, weight_decay=cfg.wd)
    total = cfg.epochs * math.ceil(len(train_loader) / cfg.accum)
    if peft is not None:
        peft.setup(total); print(f"[{tag}] {peft.describe()}")
    kd = getattr(cfg, "kd", None)
    if kd:
        t_logits = torch.from_numpy(cfg.kd_teacher_logits)
        print(f"[{tag}] distilling from {kd['teachers']} | T={kd['T']} lambda={kd['lam']}")
    warm = max(1, int(cfg.warmup * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    use_amp = resolve_amp(cfg, mode)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp and AMP_DTYPE == torch.float16)
    print(f"[{tag}] autocast {AMP_DTYPE if use_amp else 'off'} | grad scaler {scaler.is_enabled()} | groups "
          + " ".join(f"{sum(p.numel() for p in g['params'])/1e6:.2f}M@{g['lr']:g}" for g in groups))

    best, best_state, bad, gstep = -1.0, None, 0, 0
    for ep in range(cfg.epochs):
        model.train()
        t0, run = time.time(), 0.0
        opt.zero_grad(set_to_none=True)
        for i, b in enumerate(train_loader):
            with amp_context(use_amp):
                logits = forward_batch(model, b, mode)
            logits, y = logits.float(), b["y"].to(device)
            loss = loss_fn(logits, y)
            if kd:
                T = kd["T"]
                tl = t_logits[b["idx"]].to(device)
                soft = F.binary_cross_entropy_with_logits(logits / T, torch.sigmoid(tl / T)) * (T * T)
                loss = (1 - kd["lam"]) * loss + kd["lam"] * soft
            if peft is not None:
                loss = loss + peft.extra_loss()
            scaler.scale(loss / cfg.accum).backward()
            run += loss.item()
            if (i + 1) % cfg.accum == 0 or (i + 1) == len(train_loader):
                scaler.unscale_(opt)
                if peft is not None:
                    peft.after_backward(gstep)
                nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], cfg.clip)
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)
                gstep += 1
                if peft is not None:
                    peft.after_step(gstep)
            if ep == 0 and i + 1 == 20:
                sp = (time.time() - t0) / 20
                print(f"[{tag}] {sp:.2f} s/batch -> ~{sp*len(train_loader)/60:.1f} min per epoch")
        probs = predict_probs(model, dev_loader, mode, use_amp)
        thr, macro = tune_global_threshold(probs, y_dev)
        m = compute_metrics(y_dev, predict_labels(probs, thr))
        print(f"[{tag}] ep {ep+1:02d}  loss {run/len(train_loader):.4f}  dev macro-F1 {macro:.4f}  micro-F1 {m['micro_f1']:.4f}  "
              f"subset {m['subset_acc']:.4f}  thr {thr:.3f}  ({time.time()-t0:.0f}s)")
        if macro > best:
            best, bad = macro, 0
            best_state = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}
        else:
            bad += 1
            if bad >= cfg.patience:
                print(f"[{tag}] early stop"); break
    model.load_state_dict(best_state, strict=False)
    return best_state


BASE = dict(
    # long-document representation: chunk = <s> + 510 tokens + </s>
    chunk_stride=510, max_chunks=16, chunk_strategy="head_tail",
    # chunk vector = '+'-joined subset of mean | max | last
    parts="mean+max+last",
    # backbone depth: None = all 8 blocks; k = first k blocks (KD student)
    n_blocks=None, txl_chunk_bs=8,
    # chunk-level bidirectional head
    chunk_rnn="gru", chunk_hid=256, chunk_layers=1, pool="attn", dropout=0.3,
    # loss
    loss="bce_pw", pw_power=0.5, pw_max=30.0,
    # optimisation
    lr=1e-3, lr_backbone=2e-5, lr_peft=5e-4, wd=0.01, epochs=20, patience=5, batch_size=32, accum=1, warmup=0.06, clip=1.0,
    seed=42, min_support=3, amp=None,
    # adaptation: None | dict(method=..., **overrides) ; distillation: dict(teachers=[...], T=2.0, lam=0.5)
    peft=None, kd=None,
)

RESULTS = {}
for f in sorted(glob.glob(os.path.join(p5_dir, "result_*.pkl"))):          # survive a Colab disconnect
    RESULTS[os.path.basename(f)[7:-4]] = pickle.load(open(f, "rb"))
print("results already on Drive:", list(RESULTS))


def run_experiment(name, mode="cached", keep_train_logits=False, **over):
    cfg = SimpleNamespace(**{**BASE, **over})
    parts = parse_parts(cfg.parts)
    set_seed(cfg.seed)
    if cfg.peft and mode != "token":
        raise ValueError("peft adapts the backbone itself -> use mode='token'")
    if mode == "cached":
        ftr, fdv, fte = (get_features(s, e, cfg) for s, e in (("train", enc_train), ("dev", enc_dev), ("test", enc_test)))
        cols = feat_columns(parts, D_MODEL)
        d_tr, d_dv, d_te = FeatDocs(ftr, y_train), FeatDocs(fdv, y_dev), FeatDocs(fte, None)
        coll, model = make_collate_feats(cols), CachedFeatureClassifier(len(cols), L, cfg)
    else:
        d_tr, d_dv, d_te = ChunkedDocs(enc_train, y_train, cfg), ChunkedDocs(enc_dev, y_dev, cfg), ChunkedDocs(enc_test, None, cfg)
        coll, model = collate_tokens, TXLTokenClassifier(L, cfg)
    if cfg.kd:
        miss = [t for t in cfg.kd["teachers"] if "train_logits" not in RESULTS.get(t, {})]
        assert not miss, f"teacher(s) {miss}: run them first with keep_train_logits=True"
        cfg.kd_teacher_logits = np.mean([RESULTS[t]["train_logits"] for t in cfg.kd["teachers"]], axis=0).astype(np.float32)
    model.to(device)
    mk = lambda d, sh: DataLoader(d, batch_size=cfg.batch_size if sh else max(cfg.batch_size, 8), shuffle=sh,
                                  collate_fn=coll, num_workers=0)
    tr, dv, te, tr_eval = mk(d_tr, True), mk(d_dv, False), mk(d_te, False), mk(d_tr, False)
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    params_M, size_MB = model_footprint(model, mode, cfg.n_blocks)
    print(f"\n=== {name} | mode={mode} | parts={'+'.join(parts)} | blocks {cfg.n_blocks or N_BLOCKS} | trainable {n_train/1e6:.3f}M | "
          f"deployed {params_M:.1f}M params / {size_MB:.0f} MB ===")
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    train_model(model, mode, tr, dv, y_dev, cfg, y_train, name)
    train_s = time.time() - t0
    peak_mb = torch.cuda.max_memory_allocated() / 2**20 if device.type == "cuda" else 0.0
    amp_on = resolve_amp(cfg, mode)
    t0 = time.time(); lg_dev = predict_logits(model, dv, mode, amp_on); infer_s = time.time() - t0
    p_dev = 1.0 / (1.0 + np.exp(-np.clip(lg_dev, -30, 30)))
    p_test = predict_probs(model, te, mode, amp_on)
    g_thr, _ = tune_global_threshold(p_dev, y_dev)
    pc_thr = tune_per_class_threshold(p_dev, y_dev, g_thr, min_support=cfg.min_support)
    res = dict(cfg={k: v for k, v in vars(cfg).items() if k != "kd_teacher_logits"}, mode=mode, p_dev=p_dev, p_test=p_test,
               g_thr=g_thr, pc_thr=pc_thr, trainable_params=n_train, deployed_params_M=params_M, deployed_size_MB=size_MB,
               train_seconds=train_s, dev_infer_seconds=infer_s, peak_gpu_MB=peak_mb,
               m_fixed=compute_metrics(y_dev, predict_labels(p_dev, 0.5)),
               m_global=compute_metrics(y_dev, predict_labels(p_dev, g_thr)),
               m_perclass=compute_metrics(y_dev, predict_labels(p_dev, pc_thr)),
               honest=honest_threshold_eval(p_dev, y_dev))
    if getattr(model, "peft", None) is not None and model.peft.allocator is not None:
        res["adalora_ranks"] = model.peft.allocator.effective_ranks()
    if keep_train_logits:
        res["train_logits"] = predict_logits(model, tr_eval, mode, amp_on)
    RESULTS[name] = res
    pickle.dump(res, open(os.path.join(p5_dir, f"result_{name}.pkl"), "wb"))
    print(f"[{name}] dev macro-F1  thr=0.5: {res['m_fixed']['macro_f1']:.4f} | global thr {g_thr:.3f}: "
          f"{res['m_global']['macro_f1']:.4f} | per-class: {res['m_perclass']['macro_f1']:.4f} (optimistic)")
    print(f"[{name}] honest split-half macro-F1: {res['honest']}")
    print(f"[{name}] cost: train {train_s/60:.1f} min | dev inference {infer_s:.1f}s | peak GPU {peak_mb:.0f} MB")
    del model; gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return res


def write_submission(name, path, per_class=False):
    r = RESULTS[name]
    pred = predict_labels(r["p_test"], r["pc_thr"] if per_class else r["g_thr"])
    out = [dict(id=rec["id"], pred=sorted({idx2lab[int(j)] for j in np.flatnonzero(p)}, key=lambda x: (str(type(x)), x)))
           for rec, p in zip(test_recs, pred)]
    assert len(out) == len(test_recs) == len({str(o["id"]) for o in out}), "need exactly one prediction per test id"
    assert all(len(o["pred"]) >= 1 and len(set(o["pred"])) == len(o["pred"]) and all(l in lab2idx for l in o["pred"]) for o in out)
    json.dump(out, open(path, "w"), ensure_ascii=False)
    print(f"wrote {len(out)} predictions -> {path}  (avg {np.mean([len(o['pred']) for o in out]):.2f} statutes/doc)")


def make_submission_zip(name, per_class=None):
    """prediction.json at the ROOT of submission_<name>.zip. per_class=None -> per-class thresholds only if their
    honest split-half score beats the global threshold."""
    r = RESULTS[name]
    use_pc = (r["honest"]["per_class"] > r["honest"]["global"]) if per_class is None else per_class
    d = os.path.join(p5_dir, f"sub_{name}"); os.makedirs(d, exist_ok=True)
    write_submission(name, os.path.join(d, "prediction.json"), per_class=use_pc)
    zpath = os.path.join(p5_dir, f"submission_{name}.zip")
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(os.path.join(d, "prediction.json"), arcname="prediction.json")
    with zipfile.ZipFile(zpath) as z:
        assert z.namelist() == ["prediction.json"]
    print(f"  -> {zpath}")
    return zpath


def show_results():
    print(f"\n{'experiment':20s} {'macro-F1':>8s} {'micro-F1':>8s} {'subset':>7s} {'honest(g)':>9s} {'train M':>8s} "
          f"{'deploy M':>8s} {'MB':>6s} {'train min':>9s} {'dev inf s':>9s} {'peak MB':>8s}")
    for n, r in RESULTS.items():
        m = r["m_global"]
        print(f"{n:20s} {m['macro_f1']:8.4f} {m['micro_f1']:8.4f} {m['subset_acc']:7.4f} {r['honest']['global']:9.4f} "
              f"{r['trainable_params']/1e6:8.3f} {r['deployed_params_M']:8.1f} {r['deployed_size_MB']:6.0f} "
              f"{r['train_seconds']/60:9.1f} {r['dev_infer_seconds']:9.1f} {r['peak_gpu_MB']:8.0f}")

results already on Drive: ['cached_bigru', 'e1_lorma_plus']


## 11. RUN: the six Phase-5 experiments for the Transformer-XL architecture
Run the cells one at a time. Each saves `result_<name>.pkl` to Drive, so after a disconnect you re-run sections 0–10 and continue.

In [13]:
TOK = dict(mode="token", batch_size=4, accum=4, max_chunks=8, epochs=4, patience=2,
           lr=1e-3, lr_peft=5e-4, lr_backbone=2e-5, txl_chunk_bs=8)


In [13]:

# baseline: frozen Phase-1 Transformer-XL + BiGRU on cached features (zero backbone adaptation; NOT one of the six)
r_base = run_experiment("cached_bigru", mode="cached")

extracting Transformer-XL features for 'train' ...
  147,520 chunks | 64,656,697 tokens | blocks 8 | amp None | budget 24576 tok/batch
  features 3,200/147,520 chunks | 31,717 tok/s | 0.1 min | ETA 33.9 min
  features 6,400/147,520 chunks | 30,344 tok/s | 0.2 min | ETA 35.3 min
  features 9,600/147,520 chunks | 28,856 tok/s | 0.4 min | ETA 37.0 min
  features 12,800/147,520 chunks | 26,850 tok/s | 0.7 min | ETA 39.4 min
  features 15,999/147,520 chunks | 25,339 tok/s | 1.1 min | ETA 41.4 min
  features 18,927/147,520 chunks | 23,990 tok/s | 1.6 min | ETA 43.3 min
  features 21,485/147,520 chunks | 22,754 tok/s | 2.1 min | ETA 45.2 min
  features 23,786/147,520 chunks | 21,798 tok/s | 2.7 min | ETA 46.7 min
  features 25,888/147,520 chunks | 21,122 tok/s | 3.3 min | ETA 47.8 min
  features 27,834/147,520 chunks | 20,519 tok/s | 3.9 min | ETA 48.7 min
  features 29,649/147,520 chunks | 19,974 tok/s | 4.5 min | ETA 49.5 min
  features 31,355/147,520 chunks | 19,507 tok/s | 5.1 min | ETA 5

In [14]:
# E1  LoRMA+ (compulsory): multiplicative low-rank update of q_proj and v_proj in the top 4 blocks
r1 = run_experiment("e1_lorma_plus", keep_train_logits=True,
                    peft=dict(method="lorma_plus", r=8, alpha=16, top_layers=4, targets=("q_proj", "v_proj")), **TOK)


=== e1_lorma_plus | mode=token | parts=mean+max+last | blocks 8 | trainable 2.612M | deployed 78.6M params / 314 MB ===
[e1_lorma_plus] PEFT 'lorma_plus': 0.061M new parameters in 8 modules
[e1_lorma_plus] autocast off | grad scaler False | groups 2.55M@1.56006e-06 0.06M@7.80031e-07
[e1_lorma_plus] 0.95 s/batch -> ~169.2 min per epoch
[e1_lorma_plus] ep 01  loss 0.4448  dev macro-F1 0.0972  micro-F1 0.1387  subset 0.0000  thr 0.125  (10878s)
[e1_lorma_plus] ep 02  loss 0.4087  dev macro-F1 0.1352  micro-F1 0.2466  subset 0.0000  thr 0.250  (10879s)
[e1_lorma_plus] ep 03  loss 0.3803  dev macro-F1 0.1658  micro-F1 0.3009  subset 0.0007  thr 0.325  (10891s)
[e1_lorma_plus] ep 04  loss 0.3657  dev macro-F1 0.1749  micro-F1 0.3052  subset 0.0007  thr 0.325  (10893s)
[e1_lorma_plus] dev macro-F1  thr=0.5: 0.1284 | global thr 0.325: 0.1749 | per-class: 0.1882 (optimistic)
[e1_lorma_plus] honest split-half macro-F1: {'fixed_0.5': 0.12807510793209076, 'global': 0.17374567687511444, 'per_class

In [ ]:
# E2  AdaLoRA: SVD-form update on all 6 linears of the top 4 blocks, rank budget 12 -> 6 by importance
r2 = run_experiment("e2_adalora", keep_train_logits=True,
                    peft=dict(method="adalora", init_r=12, target_r=6, top_layers=4), **TOK)


=== e2_adalora | mode=token | parts=mean+max+last | blocks 8 | trainable 3.159M | deployed 79.1M params / 316 MB ===
[e2_adalora] PEFT 'adalora': 0.609M new parameters in 24 modules | AdaLoRA ranks now min 0 / mean 0.0 / max 0
[e2_adalora] autocast off | grad scaler False | groups 2.55M@1.56006e-06 0.61M@7.80031e-07
[e2_adalora] 0.84 s/batch -> ~150.0 min per epoch
[e2_adalora] ep 01  loss 0.4801  dev macro-F1 0.0754  micro-F1 0.1050  subset 0.0000  thr 0.100  (11783s)


In [14]:
# E3  QLoRA: all 8 blocks stored in 4-bit NF4, LoRA on the 6 linears of the top 4 blocks
r3 = run_experiment("e3_qlora", keep_train_logits=True,
                    peft=dict(method="qlora", r=16, alpha=32, top_layers=4), **TOK)


=== e3_qlora | mode=token | parts=mean+max+last | blocks 8 | trainable 3.361M | deployed 79.3M params / 145 MB ===
[e3_qlora] PEFT 'qlora': 0.811M new parameters in 24 modules
[e3_qlora] autocast off | grad scaler False | groups 2.55M@1.56006e-06 0.81M@7.80031e-07
[e3_qlora] 0.86 s/batch -> ~153.2 min per epoch
[e3_qlora] ep 01  loss 0.4537  dev macro-F1 0.0757  micro-F1 0.1065  subset 0.0000  thr 0.100  (11703s)
[e3_qlora] ep 02  loss 0.4422  dev macro-F1 0.0767  micro-F1 0.0889  subset 0.0000  thr 0.075  (11684s)
[e3_qlora] ep 03  loss 0.4387  dev macro-F1 0.0775  micro-F1 0.1118  subset 0.0000  thr 0.100  (11693s)
[e3_qlora] ep 04  loss 0.4370  dev macro-F1 0.0797  micro-F1 0.1126  subset 0.0000  thr 0.100  (11684s)
[e3_qlora] dev macro-F1  thr=0.5: 0.0107 | global thr 0.100: 0.0797 | per-class: 0.0856 (optimistic)
[e3_qlora] honest split-half macro-F1: {'fixed_0.5': 0.010656548663973808, 'global': 0.07963721454143524, 'per_class': 0.08553202450275421}
[e3_qlora] cost: train 779.5 

In [ ]:
# E4  Houlsby adapters after attention and FFN of the top 4 blocks
r4 = run_experiment("e4_adapter", keep_train_logits=True,
                    peft=dict(method="adapter", bottleneck=64, top_layers=4), **TOK)


=== e4_adapter | mode=token | parts=mean+max+last | blocks 8 | trainable 3.343M | deployed 79.3M params / 317 MB ===
[e4_adapter] PEFT 'adapter': 0.793M new parameters in 8 modules
[e4_adapter] autocast off | grad scaler False | groups 2.55M@1.56006e-06 0.79M@7.80031e-07
[e4_adapter] 0.91 s/batch -> ~161.9 min per epoch
[e4_adapter] ep 01  loss 0.4534  dev macro-F1 0.0765  micro-F1 0.1048  subset 0.0000  thr 0.100  (10206s)
[e4_adapter] ep 02  loss 0.4407  dev macro-F1 0.0774  micro-F1 0.1153  subset 0.0000  thr 0.100  (10198s)


In [ ]:
# E5  Partial / layer-wise fine-tuning: Phase-1 weights of the top 3 blocks + final norm are updated
r5 = run_experiment("e5_partial_ft", keep_train_logits=True, peft=dict(method="partial", k=3), **TOK)

In [ ]:
# E6  Knowledge distillation
#   teacher: ensemble (mean logit) of E1-E5 = 8 adapted blocks + hierarchical BiGRU head
#   student: the FIRST 4 Phase-1 blocks (frozen, cached features, parameter-free LayerNorm on top) + a smaller BiGRU (chunk_hid=128)
#   loss   : (1-lam) * weighted BCE + lam * T^2 * BCE(student/T, sigmoid(teacher/T))
r6 = run_experiment("e6_kd_student", mode="cached", n_blocks=4, chunk_hid=128,
                    kd=dict(teachers=["e1_lorma_plus", "e2_adalora", "e3_qlora", "e4_adapter", "e5_partial_ft"], T=2.0, lam=0.5))

In [ ]:
# ---- optional extras (uncomment) ----
# r6b = run_experiment("e6b_student_no_kd", mode="cached", n_blocks=4, chunk_hid=128)                  # isolates the KD effect
# r6c = run_experiment("e6c_student_tuned_kd", n_blocks=4, chunk_hid=128, peft=dict(method="partial", k=2),
#                      kd=dict(teachers=["e1_lorma_plus", "e5_partial_ft"], T=2.0, lam=0.5), **TOK)   # student also adapts its blocks
# r7  = run_experiment("x_prefix", peft=dict(method="prefix", n_tokens=16), **{**TOK, "lr_peft": 5e-3})  # XL memory as prefix: all blocks get grads
# r8  = run_experiment("x_prompt", peft=dict(method="prompt", n_tokens=20), **{**TOK, "lr_peft": 5e-2})  # backward through all 8 blocks: slow
# r9  = run_experiment("x_lora", peft=dict(method="lora", r=8, alpha=16, top_layers=4), **TOK)          # additive vs multiplicative (E1)
# r10 = run_experiment("x_lorma_pi", peft=dict(method="lorma_pi", r=8, top_layers=4), **TOK)
# r11 = run_experiment("x_full_ft", peft=dict(method="full"), **{**TOK, "batch_size": 2, "accum": 8})
# Representation / loss / threshold ablations on the same code (cheap in cached mode):
# run_experiment("abl_last_only", mode="cached", parts="last")          # causal model: is the </s> state enough?
# run_experiment("abl_mean_only", mode="cached", parts="mean")
# run_experiment("abl_uniform16", mode="cached", chunk_strategy="uniform")
# run_experiment("abl_label_attn", mode="cached", pool="label_attn")
# run_experiment("abl_asl", mode="cached", loss="asl")
# run_experiment("abl_overlap", mode="cached", chunk_stride=384)       # overlapping chunks

## 12. Summary, rare-statute analysis, submissions

In [ ]:
show_results()
if "e2_adalora" in RESULTS:
    print("AdaLoRA final ranks per adapted matrix:", RESULTS["e2_adalora"]["adalora_ranks"])

best_name = max(RESULTS, key=lambda n: RESULTS[n]["honest"]["global"])
r = RESULTS[best_name]
print(f"\nF1 by number of TRAIN docs per statute ({best_name}):")
for bucket, n_cls, f1 in rare_report(y_dev, predict_labels(r["p_dev"], r["g_thr"]), y_train):
    print(f"  train-freq {bucket:>8s}: {n_cls:4d} statutes  mean F1 {f1:.3f}")

# one zip per experiment (the brief needs >= 6 distinct, non-identical submissions)
for name in RESULTS:
    make_submission_zip(name)

### Report checklist (brief §11.5), with where the evidence comes from
* **Technique comparison, and does PEFT approach full fine-tuning?** Use the summary table (honest macro-F1 vs trainable params). Run `x_full_ft` for the upper reference.
* **LoRMA⁺ vs LoRA** (multiplicative vs additive at the same rank): compare E1 with `x_lora`.
* **Long documents:** try `chunk_strategy` head / tail / head_tail / uniform, `max_chunks`, and overlapping `chunk_stride`.
* **Causal backbone:** compare `parts="last"` with `mean` and `mean+max+last`. This shows how much the `</s>` state summarises a chunk.
* **Rare statutes and class imbalance:** use the F1-by-frequency buckets, and compare `loss` bce / bce_pw / asl.
* **Thresholds:** compare fixed 0.5, global, and per-class (`honest` split-half numbers; the per-class dev score is optimistic).
* **KD:** compare E6 with `e6b_student_no_kd` (the KD effect) and with the teacher (gap), plus deployed MB / dev inference time / peak GPU.
* **Phase-1 transfer:** compare the cached frozen baseline with the adapted runs. Also compare across architectures: the encoder–decoder and GQLA Phase-5 notebooks use the same pipeline.